# Detecção de doenças em folhas — treino

Treina a ConvNeXt-Tiny em etapas (PlantVillage → fotos de campo → culturas do Brasil), avalia em três testes e compara cada modificação com o seu controle. Toda a lógica está no pacote `folhas/`; este notebook só chama funções.

**Antes de rodar:** envie `deteccao_folhas.zip` para a raiz do Google Drive e selecione `Ambiente de execução → Alterar tipo → GPU T4`. Tempo total ≈ 2,5 h; se a sessão cair, reexecute tudo (retoma dos checkpoints).

## 1. Ambiente

In [ ]:
# [colab] monta o Drive, descompacta o projeto e instala a dependência que falta
import subprocess, sys, zipfile
from google.colab import drive

drive.mount('/content/drive')
zipfile.ZipFile('/content/drive/MyDrive/deteccao_folhas.zip').extractall('/content')
sys.path.insert(0, '/content/deteccao_folhas')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True)

## 2. Dados e protocolo
Baixa as bases (PlantVillage, PlantDoc, PlantWild), monta os caches e mede o vazamento treino → teste. Os valores padrão de `Config` são o protocolo completo (ver `docs/protocolo.md`); edite aqui para mudar, por exemplo `Config(epocas=4)`.

In [ ]:
from folhas import pipeline as p
from folhas.config import Config

cfg = Config()
ctx = p.preparar(cfg)

In [ ]:
p.previsualizar(ctx)   # confere o aumento de dados e a troca de fundo

## 3. PlantVillage: `baseline` (amostragem natural) e `balanceado` (classes equiprováveis)

In [ ]:
p.etapas_plantvillage(ctx)

## 4. Adaptação ao campo: `campo_pd` (+ PlantDoc) e `campo_todos` (+ PlantDoc + PlantWild)

In [ ]:
p.etapas_campo(ctx)

## 5. Culturas do Brasil: `brasil` e o controle `campo_cont`

In [ ]:
p.etapa_brasil(ctx)

## 6. Troca de fundo e ruído entre treinos

In [ ]:
p.ablacao_troca_fundo(ctx)

In [ ]:
p.sementes(ctx)

## 7. Resultados: métricas, comparações e gráfico

In [ ]:
p.resultados(ctx)

## 8. Exportar o modelo para a API
Gera `model.pt` e `model_config.json` na pasta de saídas do Drive.

In [ ]:
p.exportar(ctx)